# Notebook 04 — Conditional Price Distribution (Quantile Model)

**Across the Water: French Nuclear Unplanned Outages and GB Power Prices**

**Conditional on Notebook 02 DA gate passing.**

Fits P25 and P75 models of GB DA baseload price. Linear Quantile Regression
is the primary model; LightGBM QR is a robustness check only.

---

### Pre-registered model selection rule

1. Fit LQR on 2020–2023 training data.
2. Fit LightGBM QR on same data (early stopping, max_depth=5).
3. Evaluate both on 2024 validation data using pinball loss at P25 and P75.
4. If LGBM pinball loss > 10% lower than LQR on **both** quantiles: select LGBM.
   Otherwise: select LQR.

Rationale: LQR is more robust on ~2,000 observations. 10% threshold prevents
switching on validation noise.


> 📋 **NOT RUN** — Pre-registered but not executed.
> The DA hard gate (Notebook 02) failed before this gate was reached.
> Published unmodified to demonstrate pre-registration discipline.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.regression.quantile_regression import QuantReg
from sklearn.metrics import mean_pinball_loss
import lightgbm as lgb

from src import log, load, DATA_DIR, MAIN_START, MAIN_END
from src.utils import add_season_col

TRAIN_END = "2024-01-01"
VAL_END   = "2025-01-01"
print("Setup complete.")


---
## 1. Build Feature Matrix

In [ ]:
# Rebuild panel (notebook-independent)
gb_da = load("elexon_da_prices")[["gb_da_price"]]
gb_da.index = pd.to_datetime(gb_da.index, utc=True)

entso = load("entso_unavailability")
entso.index = pd.to_datetime(entso.index, utc=True)

def daily_outage_mw(df, t):
    s = df[df["outage_type"]==t]
    return s.groupby(s.index.normalize())["unavailable_mw"].sum().rename(f"{t}_outage_mw")

neso = load("neso_demand")
neso.index = pd.to_datetime(neso.index, utc=True)
neso_d = neso.resample("D").mean()
neso_d["wind_forecast_error"]   = neso_d.get("wind_actual_mw", np.nan) - neso_d.get("wind_forecast_mw", np.nan)
neso_d["demand_forecast_error"] = neso_d.get("demand_actual_mw", np.nan) - neso_d.get("demand_forecast_mw", np.nan)
ifa_cols = [c for c in ["ifa1_flow_mw","ifa2_flow_mw"] if c in neso_d.columns]
neso_d["ifa_flow_mw"] = neso_d[ifa_cols].sum(axis=1) if ifa_cols else np.nan

fr_temp = load("fr_temperature")[["fr_temp_deviation"]]
fr_temp.index = pd.to_datetime(fr_temp.index, utc=True)

try:
    ttf = load("ttf_spot")[["ttf_spot_eur_mwh"]]
    ttf.index = pd.to_datetime(ttf.index, utc=True)
except FileNotFoundError:
    ttf = pd.DataFrame(columns=["ttf_spot_eur_mwh"])

panel = gb_da.copy()
panel = panel.join(daily_outage_mw(entso,"unplanned"), how="left")
panel = panel.join(daily_outage_mw(entso,"planned"),   how="left")
panel = panel.join(neso_d[["wind_forecast_error","demand_forecast_error","ifa_flow_mw"]], how="left")
panel = panel.join(fr_temp, how="left")
if not ttf.empty:
    panel = panel.join(ttf, how="left")
else:
    panel["ttf_spot_eur_mwh"] = np.nan

panel["unplanned_outage_mw"] = panel["unplanned_outage_mw"].fillna(0)
panel["planned_outage_mw"]   = panel["planned_outage_mw"].fillna(0)
panel = panel[panel.index >= pd.Timestamp(MAIN_START, tz="UTC")]
panel = add_season_col(panel)
panel["year"] = panel.index.year
panel["month"] = panel.index.month
panel["dow"]   = panel.index.dayofweek

print(f"Panel: {len(panel)} rows")


---
## 2. Train/Validation Split

In [ ]:
FEATURE_COLS = [
    "unplanned_outage_mw", "planned_outage_mw",
    "wind_forecast_error", "demand_forecast_error",
    "ifa_flow_mw", "fr_temp_deviation", "ttf_spot_eur_mwh",
    "month", "dow",
]
available = [c for c in FEATURE_COLS if c in panel.columns]

# Season dummies for LQR
season_d = pd.get_dummies(panel["season"], prefix="s", drop_first=True)

train_mask = panel.index < pd.Timestamp(TRAIN_END, tz="UTC")
val_mask   = (panel.index >= pd.Timestamp(TRAIN_END, tz="UTC")) & (panel.index < pd.Timestamp(VAL_END, tz="UTC"))

y_train = panel.loc[train_mask, "gb_da_price"]
y_val   = panel.loc[val_mask,   "gb_da_price"]

X_train_raw = panel.loc[train_mask, available]
X_val_raw   = panel.loc[val_mask,   available]

# LQR design matrix (with season dummies)
X_lqr_train = pd.concat([X_train_raw, season_d[train_mask]], axis=1).astype(float)
X_lqr_val   = pd.concat([X_val_raw,   season_d[val_mask]],   axis=1).astype(float)
X_lqr_train = sm.add_constant(X_lqr_train)
X_lqr_val   = sm.add_constant(X_lqr_val)

# Drop NaN rows
mask_tr = X_lqr_train.notna().all(axis=1) & y_train.notna()
mask_va = X_lqr_val.notna().all(axis=1) & y_val.notna()

print(f"Train: {mask_tr.sum()} obs  |  Validation: {mask_va.sum()} obs")


---
## 3. Linear Quantile Regression (primary model)

In [ ]:
def fit_lqr(X_train, y_train, quantile):
    model = QuantReg(y_train[X_train.notna().all(axis=1) & y_train.notna()],
                     X_train[X_train.notna().all(axis=1) & y_train.notna()])
    return model.fit(q=quantile, max_iter=2000)

print("Fitting LQR P25...")
lqr_p25 = fit_lqr(X_lqr_train, y_train, 0.25)
print(f"  P25 pseudo-R²: {lqr_p25.prsquared:.4f}")

print("Fitting LQR P75...")
lqr_p75 = fit_lqr(X_lqr_train, y_train, 0.75)
print(f"  P75 pseudo-R²: {lqr_p75.prsquared:.4f}")

# Validation predictions
pred_lqr_p25 = lqr_p25.predict(X_lqr_val[mask_va])
pred_lqr_p75 = lqr_p75.predict(X_lqr_val[mask_va])

pb_lqr_p25 = mean_pinball_loss(y_val[mask_va], pred_lqr_p25, alpha=0.25)
pb_lqr_p75 = mean_pinball_loss(y_val[mask_va], pred_lqr_p75, alpha=0.75)

print(f"\nLQR Validation Pinball Loss:")
print(f"  P25: {pb_lqr_p25:.4f}")
print(f"  P75: {pb_lqr_p75:.4f}")


---
## 4. LightGBM Quantile Regression (robustness check)

In [ ]:
def fit_lgbm_quantile(X_train, y_train, X_val, y_val, quantile):
    """Fit LightGBM quantile regression with early stopping."""
    X_tr = X_train.dropna()
    y_tr = y_train[X_tr.index]
    X_v  = X_val.dropna()
    y_v  = y_val[X_v.index]

    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval   = lgb.Dataset(X_v,  label=y_v, reference=dtrain)

    params = {
        "objective":  "quantile",
        "alpha":       quantile,
        "max_depth":   5,
        "learning_rate": 0.05,
        "n_estimators": 500,
        "num_leaves":  20,
        "min_child_samples": 20,
        "verbosity": -1,
    }

    callbacks = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=-1)]
    model = lgb.train(
        params, dtrain,
        valid_sets=[dval],
        callbacks=callbacks,
    )
    return model

X_lgbm_train = X_train_raw.fillna(X_train_raw.median())
X_lgbm_val   = X_val_raw.fillna(X_train_raw.median())

print("Fitting LightGBM P25...")
lgbm_p25 = fit_lgbm_quantile(
    X_lgbm_train[mask_tr.values[:len(X_lgbm_train)]],
    y_train[mask_tr.values[:len(y_train)]],
    X_lgbm_val[mask_va.values[:len(X_lgbm_val)]],
    y_val[mask_va.values[:len(y_val)]],
    0.25
)
print("Fitting LightGBM P75...")
lgbm_p75 = fit_lgbm_quantile(
    X_lgbm_train[mask_tr.values[:len(X_lgbm_train)]],
    y_train[mask_tr.values[:len(y_train)]],
    X_lgbm_val[mask_va.values[:len(X_lgbm_val)]],
    y_val[mask_va.values[:len(y_val)]],
    0.75
)

pred_lgbm_p25 = lgbm_p25.predict(X_lgbm_val[mask_va.values[:len(X_lgbm_val)]])
pred_lgbm_p75 = lgbm_p75.predict(X_lgbm_val[mask_va.values[:len(X_lgbm_val)]])

pb_lgbm_p25 = mean_pinball_loss(y_val[mask_va], pred_lgbm_p25, alpha=0.25)
pb_lgbm_p75 = mean_pinball_loss(y_val[mask_va], pred_lgbm_p75, alpha=0.75)

print(f"\nLightGBM Validation Pinball Loss:")
print(f"  P25: {pb_lgbm_p25:.4f}")
print(f"  P75: {pb_lgbm_p75:.4f}")


---
## 5. Pre-registered Model Selection

In [ ]:
print("=" * 55)
print("MODEL SELECTION (pre-registered rule)")
print("=" * 55)
print()
print(f"{'Model':<12} {'P25 Pinball':>14} {'P75 Pinball':>14}")
print("-" * 42)
print(f"{'LQR':<12} {pb_lqr_p25:>14.4f} {pb_lqr_p75:>14.4f}")
print(f"{'LightGBM':<12} {pb_lgbm_p25:>14.4f} {pb_lgbm_p75:>14.4f}")
print()

lgbm_better_p25 = pb_lgbm_p25 < pb_lqr_p25 * 0.90
lgbm_better_p75 = pb_lgbm_p75 < pb_lqr_p75 * 0.90

print(f"LightGBM >10% better on P25: {'Yes' if lgbm_better_p25 else 'No'}")
print(f"LightGBM >10% better on P75: {'Yes' if lgbm_better_p75 else 'No'}")
print()

if lgbm_better_p25 and lgbm_better_p75:
    PRODUCTION_MODEL = "lgbm"
    print("→ SELECTED: LightGBM QR (>10% improvement on both quantiles)")
else:
    PRODUCTION_MODEL = "lqr"
    print("→ SELECTED: Linear Quantile Regression")
    print("  (LightGBM does not meet 10% improvement threshold on both quantiles)")
print()
print(f"Production model: {PRODUCTION_MODEL.upper()}")


---
## 6. Conditional P25 — Trading Signal

In [ ]:
# Compute conditional P25 on full panel for use in Notebook 05
X_full = pd.concat([panel[available], season_d], axis=1).astype(float)
X_full_lqr = sm.add_constant(X_full)

full_mask = X_full_lqr.notna().all(axis=1) & panel["gb_da_price"].notna()

if PRODUCTION_MODEL == "lqr":
    pred_p25_full = lqr_p25.predict(X_full_lqr[full_mask])
    pred_p75_full = lqr_p75.predict(X_full_lqr[full_mask])
else:
    X_full_lgbm = X_full.fillna(X_full.median())
    pred_p25_full = lgbm_p25.predict(X_full_lgbm[full_mask.values[:len(X_full_lgbm)]])
    pred_p75_full = lgbm_p75.predict(X_full_lgbm[full_mask.values[:len(X_full_lgbm)]])

signal_df = pd.DataFrame({
    "gb_da_price": panel.loc[full_mask, "gb_da_price"],
    "conditional_p25": pred_p25_full,
    "conditional_p75": pred_p75_full,
    "unplanned_outage_mw": panel.loc[full_mask, "unplanned_outage_mw"],
}, index=panel.index[full_mask])

signal_df["p25_above_da"] = signal_df["conditional_p25"] > signal_df["gb_da_price"]
signal_df["estimated_edge"] = signal_df["conditional_p25"] - signal_df["gb_da_price"]

signal_df.to_parquet(DATA_DIR / "signal_quantile.parquet")
print(f"Saved signal_quantile.parquet ({len(signal_df)} rows)")
print()
print("Signal summary (on outage days):")
outage_signal = signal_df[signal_df["unplanned_outage_mw"] > 0]
print(f"  Days with conditional P25 > DA price: {outage_signal['p25_above_da'].sum()} / {len(outage_signal)}")
print(f"  Mean estimated edge: £{outage_signal['estimated_edge'].mean():.3f}/MWh")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(signal_df["unplanned_outage_mw"]/1000,
           signal_df["conditional_p25"],
           alpha=0.3, s=10, color="steelblue", label="Conditional P25")
ax.scatter(signal_df["unplanned_outage_mw"]/1000,
           signal_df["gb_da_price"],
           alpha=0.15, s=8, color="grey", label="Actual DA price")
ax.set_xlabel("Unplanned outage (GW)")
ax.set_ylabel("£/MWh")
ax.set_title("Conditional P25 vs Outage Size")
ax.legend()

ax = axes[1]
ax.plot(signal_df.index, signal_df["conditional_p25"], lw=0.6, color="steelblue", label="Conditional P25")
ax.plot(signal_df.index, signal_df["gb_da_price"],     lw=0.4, color="grey",      alpha=0.5, label="Actual")
ax.set_title("Conditional P25 vs Actual GB DA Price")
ax.set_ylabel("£/MWh")
ax.legend()

plt.tight_layout()
plt.savefig(DATA_DIR / "plot_04_quantile.png", dpi=120, bbox_inches="tight")
plt.show()


---
## Summary

| Model | P25 Pinball | P75 Pinball |
|---|---|---|
| LQR | *see output* | *see output* |
| LightGBM | *see output* | *see output* |
| **Selected** | **see output** | |

Conditional P25 saved to `data/signal_quantile.parquet` for use in Notebook 05.

Proceed to **Notebook 05** (trading rule and backtest).
